---
title: Train-Test Split & Column Transformer
description: A comprehensive guide to securely partitioning data and building automated
  preprocessing pipelines that prevent data leakage and streamline model training.
author: Md Muztahid Hassan
date: '2026-05-21 17:00:00'
toc: true
execute:
  warning: false
  echo: true
order: 8
---


## Train-Test Split

Before trusting a machine learning model to make real-world decisions, we need to know if it actually *learned* the data, or if it just memorized it.

Think of training a model like a student preparing for a test. If you give the student the exact final exam to practice with, they will score 100% just by memorizing the answers—but they will fail in the real world.

To solve this, we randomly split our dataset into two parts:

* **The Training Set (Usually 70-80%):** The practice homework. The model uses this large chunk of data to learn patterns and build its mathematical rules.
* **The Testing Set (Usually 20-30%):** The final exam. This data is kept completely hidden during the training phase. We use it at the very end to test the model's accuracy on brand-new, unseen information.

**Why is this essential?**
It prevents **overfitting** (when a model memorizes the training data but fails on new data) and gives you an honest metric of how your model will perform in production. In Python, Scikit-Learn’s `train_test_split` handles this shuffling and dividing in just one line of code!

### Choosing the Right Split Ratio

There is no single rule—the perfect split depends entirely on your total dataset size:

* **Small Data (< 10,000 rows) $\rightarrow$ 70/30 or 80/20 Split:** You need a larger percentage dedicated to testing to ensure the results are statistically reliable.
* **Large Data (> 10,000 rows) $\rightarrow$ 80/20 or 90/10 Split:** 10% of a large number is still plenty of data to test with, allowing you to feed more data to the model.
* **Massive Data (Millions of rows) $\rightarrow$ 99/1 Split:** 1% of a million is still 10,000 rows! That is more than enough for a test set, leaving 99% for the model to learn from.

Let's see them in practical. You can download the dataset from <a href="covid_toy.csv" download>here</a>

In [129]:
import numpy as np
import pandas as pd

df = pd.read_csv("covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


For train-test split, we use function `train_test_split` from sklearn. It returns four outputs: `x_train`, `x_test`, `y_train`, `y_test`. We usually need to pass 4 parameters to the `train_test_split` function: `Input Features`, `Target Variable`, `Test-ratio` and `random_state`. At let's check the size of the dataset.

In [130]:
df.shape

(100, 6)

So, we can use 80-20 ratio of train-test split.

In [131]:
from sklearn.model_selection import train_test_split

input_features = df.drop('has_covid', axis=1)
target_variable = df['has_covid']
x_train, x_test, y_train, y_test = train_test_split(input_features, target_variable, test_size=0.2, random_state=42)

Now let's see the shape of the train and test dataset.

In [132]:
x_train.shape

(80, 5)

In [133]:
x_test.shape

(20, 5)

In [134]:
y_train.shape

(80,)

In [135]:
y_test.shape

(20,)

In [136]:
x_train.head()

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi


In [137]:
y_train.to_frame().head()

,has_covid
55,Yes
88,No
26,Yes
42,Yes
69,No


Here `y_train` is a pandas series. Not a dataframe.

## Column Transformer

In the real world, datasets are almost always a messy mix of numbers (like Age or Salary) and text categories (like City or Gender). Because numbers and text require completely different treatments—you need to scale numbers, but you need to encode text—you cannot apply a single rule to your entire dataset at once.

In the past, programmers had to manually slice their datasets apart, process the numbers and text separately, and then carefully glue everything back together.

The Column Transformer completely automates this. It automatically routes the correct columns to the correct tools, processes them simultaneously, and automatically stitches everything back together into a single, clean, model-ready dataset. It saves time, reduces messy code, and prevents mistakes.

Now let's see them in practical. We will again use the `covid_toy` dataset.

### Without Column Transformer (The manual approach)

In [138]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [139]:
df = pd.read_csv("covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


Let's find out if there is any missing value.

In [140]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

Here we can see that there are 10 null values. So we need to handle this and other columns should be untouched. After handling the null values in the `fever` column, we need to update to the main dataframe.

In [141]:
x_train,x_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2, random_state=42)

x_train.head()

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi


In [142]:
# adding simple imputer to fever col
si = SimpleImputer()
x_train_fever = si.fit_transform(x_train[['fever']])
# also the test data
x_test_fever = si.transform(x_test[['fever']])
x_train_fever.shape

(80, 1)

In [143]:
x_train_fever[:10]

array([[101.],
       [100.],
       [100.],
       [100.],
       [103.],
       [103.],
       [102.],
       [101.],
       [101.],
       [101.]])

Again, we need to convert it to dataframe and add it to the main dataframe.

There are mix of different columns in the real world datasets. So we need to feature engineer every column separately. Like `age` doesn't need ordinal or nominal encoding as its value is numeric. So we need to handle this column in a separate way.
The `gender` and `city` columns needs `one-hot encoding` and the `cough` column needs `ordinal` encoding. So we can feel that the manual approach is very time consuming.

In [144]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough = oe.fit_transform(x_train[['cough']])

# also the test data
x_test_cough = oe.transform(x_test[['cough']])

x_train_cough.shape

(80, 1)

In [145]:
x_train_cough[:10]

array([[0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.]])

Now we will use `one-hot encoding` on the `gender` and `city` columns.

In [146]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])

# also the test data
x_test_gender_city = ohe.transform(x_test[['gender','city']])

x_train_gender_city.shape

(80, 4)

In [147]:
x_train_gender_city[:5]

array([[0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [1., 1., 0., 0.],
       [0., 1., 0., 0.]])

Now we have to add them to the main dataset.

In [148]:
# Extracting Age
x_train_age = x_train.drop(columns=['gender','fever','cough','city']).values
x_train_age[:10]

array([[81],
       [ 5],
       [19],
       [27],
       [73],
       [70],
       [49],
       [51],
       [64],
       [83]])

In [149]:
# also the test data
x_test_age = x_test.drop(columns=['gender','fever','cough','city']).values

x_train_age.shape

(80, 1)

In [ ]:
x_train_transformed = np.concatenate((x_train_age,x_train_fever,x_train_gender_city,x_train_cough),axis=1)
# also the test data
x_test_transformed = np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)

x_train_transformed.shape

((80, 7), (80, 5))

In [151]:
x_train_transformed[:5]

array([[ 81., 101.,   0.,   0.,   0.,   1.,   0.],
       [  5., 100.,   0.,   0.,   1.,   0.,   0.],
       [ 19., 100.,   0.,   0.,   1.,   0.,   0.],
       [ 27., 100.,   1.,   1.,   0.,   0.,   0.],
       [ 73., 103.,   0.,   1.,   0.,   0.,   0.]])

So, we can see that the manual approach is very time consuming and error prone. We can automate this process using `ColumnTransformer`.

### With Column Transformer (The automated approach)
Here is a template of how to use `ColumnTransformer`:

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# 1. Define the transformer
# It automatically encodes gender/city and passes age, fever, and cough through untouched
transformer = ColumnTransformer(
    transformers=[
        ('Name what you want', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city'])
    ],
    remainder='passthrough' # use 'drop' if you want to drop the other columns instead of passing them through
)

# 2. Fit and transform the training data (replaces np.concatenate entirely)
x_train_transformed = transformer.fit_transform(x_train)

# 3. ONLY transform the testing data
x_test_transformed = transformer.transform(x_test)
```

Now let's see them in practical. 

In [152]:
from sklearn.compose import ColumnTransformer

In [154]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [156]:
new_data = transformer.fit_transform(x_train)
new_data.shape

(80, 7)

Look it's like a magic! We have transformed the dataset in just 3 lines of code. The `ColumnTransformer` automatically handled the one-hot encoding for the specified columns and passed the rest through without any extra work from us. This not only saves time but also ensures that we don't accidentally mess up our data by forgetting to concatenate or misaligning columns.

In [157]:
new_data[:5]

array([[101.,   0.,   0.,   0.,   0.,   1.,  81.],
       [100.,   0.,   0.,   0.,   1.,   0.,   5.],
       [100.,   0.,   0.,   0.,   1.,   0.,  19.],
       [100.,   0.,   1.,   1.,   0.,   0.,  27.],
       [103.,   0.,   0.,   1.,   0.,   0.,  73.]])